# 02 — Embedding exploration

Looks at the vector index built by `just embed` and asks whether it behaves the
way [ADR 0007](../docs/adr/0007-multi-channel-embedding.md) assumes it does.

Nothing here is a test — tests live in `tests/` and run offline. This notebook
needs the real artifacts:

```sh
just setup
just ingest && just embed
```

Three questions:

1. Do the channels hold what we think they hold?
2. Does the geometry carry meaning — do cards of the same creature type sit
   closer together than cards of different types?
3. Does retrieval actually retrieve — does querying a keyword surface the cards
   whose text contains it, and does a flavor-phrased query find flavour?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from mtg_rag.corpus import real_cards
from mtg_rag.embed.config import CHANNELS, MODEL_ID, VECTOR_DIR_NAME
from mtg_rag.embed.encoder import QwenEncoder
from mtg_rag.ingest.config import CORPUS_NAME
from mtg_rag.store.chroma import open_client, open_collection, search

DATA = Path("..") / "data"

frame = pl.read_parquet(DATA / CORPUS_NAME)
cards = real_cards(frame)
client = open_client(DATA / VECTOR_DIR_NAME)

print(f"corpus       {frame.height:>7,} rows")
print(f"real cards   {cards.height:>7,}")
for channel in CHANNELS:
    print(f"  {channel:<10s} {open_collection(client, channel).count():>7,} vectors")

## 1. What the channels hold

Collection sizes differ by design: every real card has a type line, almost all
have oracle text, and only about 59% have flavour text. A card with no text in a
channel has **no entry** there rather than a zero vector
([ADR 0014](../docs/adr/0014-absent-channel-no-vector.md)).

In [ ]:
coverage = pl.DataFrame(
    {
        "channel": list(CHANNELS),
        "vectors": [open_collection(client, ch).count() for ch in CHANNELS],
    }
).with_columns((pl.col("vectors") / cards.height * 100).round(1).alias("% of real cards"))

coverage

In [ ]:
# The encoder is only needed to embed *queries* from here on.
encoder = QwenEncoder()
print(MODEL_ID, "->", encoder.dim, "dims")

In [ ]:
def vectors_for(channel: str, ids: list[str]) -> tuple[np.ndarray, list[str]]:
    """Fetch stored vectors by oracle_id, keeping the order asked for."""
    stored = open_collection(client, channel).get(ids=ids, include=["embeddings"])
    position = {card_id: i for i, card_id in enumerate(stored["ids"])}
    embeddings = np.asarray(stored["embeddings"], dtype=np.float32)
    kept = [card_id for card_id in ids if card_id in position]
    return np.stack([embeddings[position[card_id]] for card_id in kept]), kept


def sample_ids(pattern: str, limit: int = 80) -> list[str]:
    """oracle_ids of real cards whose type line contains `pattern`."""
    return (
        cards.filter(pl.col("type_line").str.contains(pattern)).head(limit)["oracle_id"].to_list()
    )

## 2. Does the geometry carry meaning?

The claim under test: cards sharing a creature type should sit closer together
in the **type** channel than cards of different types. If that does not hold,
the channel is not encoding what its name suggests.

Vectors are already unit-norm (the encoder normalises), so a dot product *is*
cosine similarity.

In [ ]:
CREATURE_TYPES = ["Goblin", "Angel", "Zombie", "Elf", "Dragon", "Merfolk"]

groups = {}
for creature in CREATURE_TYPES:
    vectors, _ = vectors_for("type", sample_ids(creature))
    groups[creature] = vectors

similarity = np.zeros((len(CREATURE_TYPES), len(CREATURE_TYPES)))
for i, a in enumerate(CREATURE_TYPES):
    for j, b in enumerate(CREATURE_TYPES):
        sims = groups[a] @ groups[b].T
        if i == j:
            # Exclude the diagonal: a card is trivially identical to itself.
            n = sims.shape[0]
            similarity[i, j] = (sims.sum() - np.trace(sims)) / (n * n - n)
        else:
            similarity[i, j] = sims.mean()

within = np.diag(similarity).mean()
across = similarity[~np.eye(len(CREATURE_TYPES), dtype=bool)].mean()
print(f"mean similarity within a type : {within:.3f}")
print(f"mean similarity across types  : {across:.3f}")
print(f"separation                    : {within - across:+.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(similarity, cmap="viridis")
ax.set_xticks(range(len(CREATURE_TYPES)), CREATURE_TYPES, rotation=45, ha="right")
ax.set_yticks(range(len(CREATURE_TYPES)), CREATURE_TYPES)
ax.set_title("Mean cosine similarity, type channel")
for i in range(len(CREATURE_TYPES)):
    for j in range(len(CREATURE_TYPES)):
        ax.text(
            j,
            i,
            f"{similarity[i, j]:.2f}",
            ha="center",
            va="center",
            color="white" if similarity[i, j] < similarity.max() * 0.75 else "black",
            fontsize=8,
        )
fig.colorbar(image, ax=ax, shrink=0.8)
fig.tight_layout()

The diagonal should dominate its row. Off-diagonal warmth is not noise and is
worth reading: creature types that share tribal or colour identity genuinely do
sit nearer each other.

In [ ]:
def pca_2d(matrix: np.ndarray) -> np.ndarray:
    """Two leading principal components — plain SVD, no extra dependency."""
    centred = matrix - matrix.mean(axis=0)
    _, _, components = np.linalg.svd(centred, full_matrices=False)
    return centred @ components[:2].T


stacked = np.concatenate([groups[c] for c in CREATURE_TYPES])
projected = pca_2d(stacked)

fig, ax = plt.subplots(figsize=(7, 6))
offset = 0
for creature in CREATURE_TYPES:
    size = len(groups[creature])
    chunk = projected[offset : offset + size]
    ax.scatter(chunk[:, 0], chunk[:, 1], label=creature, alpha=0.65, s=18)
    offset += size
ax.set_title("Type-channel vectors, first two principal components")
ax.legend(markerscale=1.5, fontsize=9)
fig.tight_layout()

## 3. Does retrieval retrieve?

Queries are embedded with `encode_queries`, which applies the model's
`Instruct:` prompt — documents were embedded with an empty prompt. Both sides
must honour that asymmetry or the geometry silently mismatches
([ADR 0012](../docs/adr/0012-embedding-model.md)).

In [ ]:
def retrieve(query: str, channel: str = "oracle", k: int = 10) -> pl.DataFrame:
    query_vector = encoder.encode_queries([query])[0]
    hits = search(open_collection(client, channel), query_vector, allow_ids=None, n_results=k)

    found = cards.filter(pl.col("oracle_id").is_in(list(hits)))
    by_id = {row["oracle_id"]: row for row in found.iter_rows(named=True)}
    # `hits` iterates nearest-first — that ordering is the payload (ADR 0008).
    return pl.DataFrame(
        [
            {
                "rank": rank,
                "distance": round(distance, 4),
                "name": by_id[card_id]["name"],
                "type_line": by_id[card_id]["type_line"],
                "oracle_text": (by_id[card_id]["oracle_text"] or "").replace("\n", " / ")[:90],
            }
            for rank, (card_id, distance) in enumerate(hits.items())
            if card_id in by_id
        ]
    )


retrieve("trample", channel="oracle", k=10)

### The keyword check

A mechanical query is the honest test of the oracle channel, because the answer
is checkable: how many of the top hits literally contain the word? Compare that
against the base rate in the corpus — that difference is the channel doing work.

In [ ]:
def keyword_precision(keyword: str, k: int = 25) -> None:
    hits = retrieve(keyword, channel="oracle", k=k)
    matched = hits["oracle_text"].str.to_lowercase().str.contains(keyword.lower()).sum()

    base = (
        cards.filter(pl.col("oracle_text").is_not_null())["oracle_text"]
        .str.to_lowercase()
        .str.contains(keyword.lower())
        .mean()
    )
    print(
        f"{keyword:<12s} top-{k}: {matched:>2}/{k} contain it  ({matched / k:.0%})"
        f"   corpus base rate: {base:.2%}"
    )


for keyword in ["trample", "flying", "lifelink", "deathtouch", "connive"]:
    keyword_precision(keyword)

### Flavour, which is the point

Mechanical recall is measurable; flavour is what this recommender exists for and
is left to human judgement ([ADR 0006](../docs/adr/0006-eval-measures-retrieval-recall.md)).
Read these and decide whether they feel right.

In [ ]:
for query in [
    "a spooky graveyard theme",
    "noble knights defending a kingdom",
    "goblins causing chaos and mayhem",
]:
    print(f"\n=== {query} ===")
    display(retrieve(query, channel="flavor", k=5)[["rank", "name", "type_line"]])

### The same query across all three channels

Each channel is a different register of language, so the same words land
differently. This is the argument for fusing rankings rather than searching one
surface ([ADR 0008](../docs/adr/0008-rrf-fusion-not-raw-scores.md)).

In [ ]:
QUERY = "undead creatures rising from the grave"

for channel in CHANNELS:
    print(f"\n=== {channel} ===")
    display(retrieve(QUERY, channel=channel, k=5)[["rank", "name", "type_line"]])